In [ ]:
!pip install -U numpy==1.26.4
!pip install -U transformers==4.41.2 accelerate sentencepiece
!pip install -U langchain==0.2.16 langchain-community==0.2.16 langchain-core==0.2.38

In [ ]:
from transformers import pipeline

In [ ]:
#testing hugging face
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=100,
    temperature=0.7
    )
print(pipe("Explain AI in education simply")[0]["generated_text"])


# Part A: Define the Use Case

For this project, I have chosen the education domain as the use case. This decision was made because education provides a strong and practical environment to demonstrate advanced multi-agent AI capabilities such as retrieval, reasoning, memory, and tool usage. The project focuses on building a multi-agent AI learning assistant designed to support students in understanding academic content more effectively. The system helps users study complex topics by retrieving relevant information from learning materials, generating simplified explanations, and producing practice questions for revision. Students often face challenges in understanding lectures notes, textbooks, and large volumes of study material. This system addresses this problem by acting as an intelligent AI tutor that can analyze questions, retrieve relevant information from multiple sources, and respond in a structured and educational manner.

# Part B: Build a Basic Chatbot

In [ ]:
#libraries
!pip install langchain==0.2.16 langchain-community==0.2.16 langchain-core==0.2.38

In [ ]:
#imports
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain_community.llms import HuggingFacePipeline

In [ ]:
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=80,
    temperature=0.5,
    do_sample=True,
    return_full_text=False   # 🔥 IMPORTANT
)
llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
#Creating prompt template
prompt = PromptTemplate(
     input_variables=["question"],
     template="""
     You are a helpful AI tutor.

    Only answer the question.
    Do Not repeat the question or instructions.

    Question:{question}
     Answer:
     """
)

In [ ]:
#Creating LLMChain
chain = LLMChain(llm=llm, prompt=prompt)

In [ ]:
#cleaning
def clean(text):
  if "Answer:" in text:
   return text.split("### Answer:")[-1].strip()
   return text.strip()

In [ ]:
#Testing chatbot
question = [
    "What is artificial intelligence",
    "Explain the role of AI in education",
    "What is machine learning?"
]

for q in question:
  response = chain.invoke({"question":q})
  answer = clean(response["text"])

  print("=" * 50)
  print("\nQ:", q)
  print("A:", response["text"])

# Part C: Add Memory

In [ ]:
#Importing
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain


In [ ]:
memory = ConversationBufferMemory()

In [ ]:
conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=True
)

In [ ]:
conversation.invoke("My name is Waleed")
conversation.invoke("What is my name?")

# Part D: Add Retrieval

In [ ]:
from langchain.docstore.document import Document
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain

In [ ]:
documents = [
    Document(page_content="Artificial Intelligence is the simulation of human intelligence in machines."),
    Document(page_content="Machine Learning is a subset of AI that allows systems to learn from data."),
    Document(page_content="In education, AI helps with personalized learning, automated grading, and tutoring systems."),
    Document(page_content="Deep learning uses neural networks with many layers to analyze complex data.")
]

In [ ]:
splitter = CharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

chunks = splitter.split_documents(documents)

In [ ]:
##!pip uninstall -y transformers sentence-transformers langchain langchain-community

##!pip install transformers==4.41.2
##!pip install sentence-transformers==2.7.0
##!pip install langchain==0.2.16 langchain-community==0.2.16

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
pip install faiss-cpu

In [ ]:
vectorstore = FAISS.from_documents(chunks, embeddings)

In [ ]:
retriever = vectorstore.as_retriever()

In [ ]:

rag_prompt = PromptTemplate(
    input_variables=["question"],
    template="""


Question:
{question}

Answer:
"""
)

In [ ]:
rag_chain = LLMChain(
    llm=llm,
    prompt=rag_prompt
)

In [ ]:
def ask_rag(question):
    docs = retriever.get_relevant_documents(question)

    context = "\n".join([d.page_content for d in docs])

    result = rag_chain.invoke({  "question": question })

    return result["text"]

In [ ]:
print(ask_rag("What is machine learning?"))

# Part E: Add Multiple Agents

In [ ]:
##!pip install langgraph langchain langchain-community

In [ ]:
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain.chains import LLMChain

In [ ]:
planner_prompt = PromptTemplate(
    input_variables=["question"],
    template="""
You are a classifier.

Return ONLY one word:
rag or llm

Question: {question}
"""
)
planner_chain = LLMChain(
    llm=llm,
    prompt=planner_prompt
)

In [ ]:
def retrieve_context(question):
  docs = retriever.get_relevant_documents(question)
  return "\n".join([d.page_content for d in docs])

In [ ]:
answer_prompt = PromptTemplate(
    input_variables=["question", "context"],
    template="""

 Question:
  {question}

    Context:
    {context}


    """
)

answer_chain = LLMChain(llm=llm, prompt=answer_prompt)

In [ ]:
def multi_agent_system(question):
  decision = planner_chain.invoke({"question": question})["text"].strip()

  if "rag" in decision:
    context = retrieve_context(question)
    result = answer_chain.invoke({
        "question": question,
        "context": context
    })
    return result["text"]
  else:
    result = llm.invoke(question)
    return result

In [ ]:
response = rag_chain.invoke({
    "question": "What is machine learning?"
})

In [ ]:
def ask(question):
    context = retrieve_context(question)

    return rag_chain.invoke({
        "question": question,
        "context": context
    })

In [ ]:
print(ask("What is machine learning?"))
print(ask("How does machine learning work in personalized learning?"))

# Part F: Add LangGraph

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph
from langgraph.graph import StateGraph, END

In [ ]:
#Defining state

class state(TypedDict):
  question: str
  context: str
  decision: str
  answer:str

In [ ]:
#planner node

def planner(state: state):
    question = state["question"]

    result = planner_chain.invoke({"question": question})

    decision = result["text"].strip().lower()


    if "rag" in decision:
        decision = "rag"
    else:
        decision = "llm"

    return {
        "question": question,
        "decision": decision
    }

In [ ]:
#retriever node

def retriever_node(state: state):
    question = state["question"]

    if isinstance(question, dict):
        question = question["question"]

    docs = retriever.get_relevant_documents(str(question))

    context = "\n".join([d.page_content for d in docs])

    return {"context": context}


In [ ]:
def answer_node(state: state):
    question = state["question"]
    context = state["context"]

    prompt = f"""
You are an AI tutor.

Answer ONLY the question in 1-2 sentences.
Do NOT generate extra questions.
Stop after the answer.

Context:
{context}

Question:
{question}

Answer:
"""

    result = llm.invoke(prompt)


    answer = result.split("Question:")[0].strip()

    return {"answer": answer}

In [ ]:
#direct llm node
def direct_llm_node(state: state):
    question = state["question"]

    prompt = f"""
Answer this clearly in 1-2 sentences:

{question}
"""

    result = llm.invoke(prompt)

    return {"answer": str(result)}

In [ ]:
#router function
def route(state: state):
  if "rag" in state["decision"]:
    return "retriever"
  else:
      return "direct"

In [ ]:
#build langgraph
graph = StateGraph(state)

In [ ]:
#adding nodes
graph.add_node("planner", planner)
graph.add_node("retriever", retriever_node)
graph.add_node("answer", answer_node)
graph.add_node("direct", direct_llm_node)

In [ ]:
#defining flow
graph.set_entry_point("planner")

graph.add_conditional_edges(
    "planner",
    route,
    {
        "retriever": "retriever",
        "direct": "direct"
    }
)

graph.add_edge("retriever", "answer")
graph.add_edge("answer", END)
graph.add_edge("direct", END)

In [ ]:
#compiling graph
app = graph.compile()

In [ ]:
response = app.invoke({
    "question": "What is machine learning?",
    "context": "",
    "decision": "",
    "answer": ""
})

print(response)

# Part G: Add MCP Servers

In [ ]:
#imports
import requests
from langgraph.graph import StateGraph


In [ ]:
#Creating API tool
def weather_api(city):
  url = f"https://wttr.in/{city}?format=3"
  response = requests.get(url)
  return response.text

In [ ]:
#Create API agent
def api_agent(state):
  question = state["question"]

  if "weather" in question.lower():
    result = weather_api("Abu Dhabi")
    return{"answer": result}

    return {"answer" "No API needed"}


In [ ]:
#Update Planner
planner_prompt + PromptTemplate(
    input_variables=["question"],
    template="""
    You are a planner agent.

    Decide:
    - "rag" → if knowledge question
- "api" → if real-time info (weather, news)
- "llm" → general question

Question: {question}

Answer ONLY one word: rag, api, or llm
"""
)

In [ ]:
#update router
def route(state):
    q = state["question"].lower()

    if "weather" in q:
        return "weather"

    if "machine learning" in q or "ai" in q:
        return "llm"

    return "llm"

In [ ]:
def weather_node(state):
    return {
        "answer": "☀️ Real weather API needed here (e.g., OpenWeatherMap)"
    }

In [ ]:
#Add API Node to graph
graph.add_node("api", api_agent)

In [ ]:
#text mcp system
result1 = app.invoke({"question": "What is machine learning?"})
print(result1["answer"])

result2 = app.invoke({"question": "What is the weather today?"})
print(result2["answer"])

# Part H: Add External APIs

In [ ]:
#imports
def books_api(query):
  url = f"https://www.googleapis.com/books/v1/volumes?q={query}"
  response = requests.get(url).json()

  results = []

  for item in response.get("items", [])[:3]:
    title = item["VolumeInfo"].get("title", "No title")
    authors = item["VolumeInfo"].get("authors", ["No author"])
    results.append(f"{title} by {', '.join(authors)}")

    return "\n".join(results)

In [ ]:
#update API Agent
def api_agent(state):
  question = state["question"].lower()

  if "weather" in question:
    return {"answer": weather_api("Abu Dhabi")}

  elif "book" in question or "study material" in question:
    return{"answer": books_api(question)}

    return{"answer": "No API used"}

In [ ]:
planner_prompt = PromptTemplate(
    input_variables=["question"],
    template="""
You are a Planner agent.

Decide the best tool:
- "rag" → for knowledge from documents
- "weather_api" → for weather-related queries
- "books_api" → for books, study materials
- "llm" → for general conversation

Question:{question}

Answer ONLY one word: rag, weather_api, books_api, or llm
"""
)

In [ ]:
#update router
def router(state):
    decision = state["decision"]

    if decision == "rag":
        return "rag_node"
    elif decision == "api":
        return "api_node"
    else:
        return "llm_node"

In [ ]:
#Split API Nodes
def weather_node(state):
  return {"answer": weather_api("Abu Dhabi")}

def books_node(state):
  return{"answer": books_api(state["question"])}

In [ ]:
#adding nodes
graph.add_node("weather", weather_node)
graph.add_node("books", books_node)

In [ ]:
print(app.invoke({"question": "What is machine learning?"})["answer"])


In [ ]:
print(app.invoke({"question": "What is the weather today?"})["answer"])

In [ ]:
print(app.invoke({"question": "Recommend books for learning AI"})["answer"])